In [ ]:
!pip -q install -U openai chromadb gradio python-dotenv tiktoken

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv(override=True)
assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY."

CHAT_MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-large"

KB_ROOT = Path("kb_pharma")

# ✅ Put DB inside this folder (simple)
DB_DIR = Path("pharma_vector_db")
DB_DIR.mkdir(parents=True, exist_ok=True)

COLLECTION_NAME = "pharma_chunks"

openai = OpenAI()
chroma = chromadb.PersistentClient(path=str(DB_DIR))

print("CWD:", Path.cwd())
print("KB_ROOT:", KB_ROOT.resolve(), "exists:", KB_ROOT.exists())
print("DB_DIR:", DB_DIR.resolve(), "exists:", DB_DIR.exists())

In [ ]:
def load_kb_documents(kb_root: Path):
    docs = []
    overview = kb_root / "company_overview.md"
    if overview.exists():
        docs.append({"source": overview.as_posix(), "type": "company", "text": overview.read_text(encoding="utf-8")})

    drugs_dir = kb_root / "drugs"
    if drugs_dir.exists():
        for file in drugs_dir.rglob("*.md"):
            docs.append({"source": file.as_posix(), "type": "drug", "text": file.read_text(encoding="utf-8")})
    return docs

documents = load_kb_documents(KB_ROOT)
print(f"Loaded {len(documents)} documents")
for d in documents:
    print("-", d["type"], d["source"])
assert documents, "No documents found."

In [ ]:
def chunk_text(text: str, chunk_size: int = 1200, overlap: int = 200):
    chunks = []
    i = 0
    while i < len(text):
        chunks.append(text[i:i+chunk_size])
        i += chunk_size - overlap
    return chunks

chunks = []
for doc in documents:
    parts = chunk_text(doc['text'])
    for idx, part in enumerate(parts):
        chunks.append({
            'id': f"{doc['source']}::chunk{idx}",
            'text': part,
            'metadata': {'source': doc['source'], 'type': doc['type'], 'chunk': idx}
        })

print(f'Created {len(chunks)} chunks')
print('\nSample chunk metadata:', chunks[0]['metadata'])
print('\nSample chunk text (first 200 chars):\n', chunks[0]['text'][:200])

In [ ]:
def reset_collection(client: chromadb.PersistentClient, name: str):
    try:
        client.delete_collection(name)
    except Exception:
        pass
    return client.get_or_create_collection(name)

collection = reset_collection(chroma, COLLECTION_NAME)

BATCH = 64
for i in range(0, len(chunks), BATCH):
    batch = chunks[i:i+BATCH]
    texts = [c['text'] for c in batch]
    ids = [c['id'] for c in batch]
    metas = [c['metadata'] for c in batch]

    embs = openai.embeddings.create(model=EMBED_MODEL, input=texts).data
    vectors = [e.embedding for e in embs]

    collection.add(ids=ids, documents=texts, metadatas=metas, embeddings=vectors)

print('✅ Stored chunks in Chroma:', collection.count())
print('DB saved at:', DB_DIR)

In [ ]:
TOP_K = 6

def retrieve(query: str, k: int = TOP_K):
    qvec = openai.embeddings.create(model=EMBED_MODEL, input=[query]).data[0].embedding
    res = collection.query(query_embeddings=[qvec], n_results=k)
    out = []
    for doc, meta in zip(res['documents'][0], res['metadatas'][0]):
        out.append({'text': doc, 'metadata': meta})
    return out

test_query = 'What does ibuprofen treat and what are common side effects?'
hits = retrieve(test_query)
print('Top hits:')
for h in hits:
    print('-', h['metadata']['source'], '#', h['metadata']['chunk'])

In [ ]:
SYSTEM = (
    "You are PharmAssist, a concise pharmaceutical knowledge worker.\n"
    "Use ONLY the provided context to answer.\n"
    "If the answer is not in the context, say: \"I don't know based on the provided notes.\" \n"
    "Be clear and concise.\n"
    "Always include citations as (source: <file>#<chunk>).\n"
    "Do not provide medical diagnosis. Encourage consulting a licensed clinician/pharmacist for personalized advice."
)

def build_context(hits):
    lines = []
    for h in hits:
        meta = h["metadata"]
        src = meta.get("source", "unknown")
        ck = meta.get("chunk", "?")
        lines.append(f"Extract from {src}#{ck}:\n{h['text']}")
    return "\n\n".join(lines) if lines else "No context found."

In [ ]:
import gradio as gr

def chat(user_message, history):
    # history is list of (user, assistant) tuples in older gradio
    hits = retrieve(user_message, k=TOP_K)
    context = build_context(hits)

    messages = [
        {"role": "system", "content": SYSTEM + "\n\nContext:\n" + context}
    ]

    # convert tuple history -> OpenAI message format
    for user, assistant in history:
        messages.append({"role": "user", "content": user})
        messages.append({"role": "assistant", "content": assistant})

    messages.append({"role": "user", "content": user_message})

    resp = openai.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        temperature=0.2
    )
    return resp.choices[0].message.content

gr.ChatInterface(chat, title="PharmAssist — RAG over kb_pharma").launch(inbrowser=True)